In [1]:
!pip install medpy nibabel pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 20.9 MB/s eta 0:00:00
  Created wheel for medpy: filename=MedPy-0.5.2-py3-none-any.whl size=224710 sha256=f7ef41d287b4d314d65fc605820c1fb5b2c938a16e0f2c03763348621a17c09d
  Stored in directory: /root/.cache/pip/wheels/89/5a/f8/b3def53b9c2133d2f8698ea2173bb5df63bd8e761ce8e9aec9
Successfully built medpy


## 1. Giải nén `nnUNet_ra`w & `nnUNet_preprocessed`

In [2]:
import os
import zipfile
from tqdm import tqdm

RAW_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_raw/Dataset101_BraTS2020.zip"
PRE_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_preprocessed/Dataset101_BraTS2020.zip"

RAW_DIR = "/content"
PRE_DIR = "/content"

def unzip(zip_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        for f in tqdm(z.namelist()):
            z.extract(f, out_dir)

In [3]:
unzip(RAW_ZIP, RAW_DIR)

100%|██████████| 1848/1848 [04:59<00:00,  6.18it/s]


In [4]:
unzip(PRE_ZIP, PRE_DIR)

100%|██████████| 2590/2590 [05:52<00:00,  7.36it/s]


## 2. Set biến môi trường nnU-Net v2

In [5]:
import os

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

nnUNet_raw: /content/nnUNet_raw
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/nnUNet_results


## 3. Tính HD95

### 3.1 Baseline

In [9]:
import numpy as np
import nibabel as nib
import pandas as pd
from medpy.metric.binary import hd95
from tqdm import tqdm

# Đường dẫn folder chứa nhãn gốc (Ground Truth) trên Drive
# Folder này chứa các file: BraTS20_Training_001.nii.gz, ...
GT_FOLDER = "/content/nnUNet_raw/Dataset101_BraTS2020/labelsTr"

# Đường dẫn folder kết quả dự đoán (Prediction) trên Drive
# Đây là folder backup bạn vừa tạo xong
PRED_FOLDER = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres/crossval_results_folds_0_1_2_3_4"

# Nơi lưu file CSV kết quả (Lưu ngay tại folder dự đoán cho tiện)
OUTPUT_CSV = os.path.join(PRED_FOLDER, "BraTS_HD95_Metrics.csv")

In [10]:
def compute_hd95_safe(pred, gt, spacing):
    """
    Tính HD95 an toàn.
    Nếu 1 trong 2 mask rỗng (không có u), HD95 không xác định (vô cực).
    Trong các cuộc thi BraTS, trường hợp này thường được gán giá trị phạt tối đa (373.13mm).
    """
    # Nếu cả 2 đều rỗng -> Khoảng cách bằng 0 (Đúng tuyệt đối)
    if np.sum(pred) == 0 and np.sum(gt) == 0:
        return 0.0

    # Nếu 1 trong 2 rỗng -> Sai hoàn toàn -> Gán giá trị phạt lớn (Max Distance)
    # Đường chéo hộp sọ người khoảng 373.13 mm
    # Điều này có thể xảy ra với 1 số ca không có ET, mà model lỡ dự đoán có ET. Đối với cuộc thi BRats là rank-> scor
    if np.sum(pred) == 0 or np.sum(gt) == 0:
        return 373.13

    try:
        # voxelspacing: Quan trọng để đổi từ pixel sang mm
        return hd95(pred, gt, voxelspacing=spacing)
    except Exception as e:
        print(f"Lỗi tính HD95: {e}")
        return np.nan

def process_case(case_id):
    try:
        gt_path = os.path.join(GT_FOLDER, f"{case_id}.nii.gz")
        pred_path = os.path.join(PRED_FOLDER, f"{case_id}.nii.gz")

        if not os.path.exists(gt_path):
            print(f"Thiếu GT: {case_id}")
            return None
        if not os.path.exists(pred_path):
            print(f"Thiếu Pred: {case_id}")
            return None

        # Load NIfTI
        gt_nii = nib.load(gt_path)
        pred_nii = nib.load(pred_path)

        # Lấy spacing (kích thước voxel theo mm) từ header
        # Thường là (1.0, 1.0, 1.0) hoặc xấp xỉ
        zoom = gt_nii.header.get_zooms()

        gt_data = gt_nii.get_fdata().astype(np.uint8)
        pred_data = pred_nii.get_fdata().astype(np.uint8)

        # --- Tách vùng (Region Extraction) ---
        # Label 1: NCR/NET, 2: ED, 3: ET

        # WT (Whole Tumor): Label 1 + 2 + 3
        gt_WT = (gt_data > 0)
        pred_WT = (pred_data > 0)

        # TC (Tumor Core): Label 1 + 3
        gt_TC = np.logical_or(gt_data == 1, gt_data == 3)
        pred_TC = np.logical_or(pred_data == 1, pred_data == 3)

        # ET (Enhancing Tumor): Label 3
        gt_ET = (gt_data == 3)
        pred_ET = (pred_data == 3)

        # --- Tính HD95 ---
        h_WT = compute_hd95_safe(pred_WT, gt_WT, zoom)
        h_TC = compute_hd95_safe(pred_TC, gt_TC, zoom)
        h_ET = compute_hd95_safe(pred_ET, gt_ET, zoom)

        return {
            "Case_ID": case_id,
            "HD95_WT": h_WT,
            "HD95_TC": h_TC,
            "HD95_ET": h_ET
        }

    except Exception as e:
        print(f"Error {case_id}: {e}")
        return None

In [11]:
if __name__ == "__main__":
    print(f"Đang đọc dữ liệu từ Drive...")
    print(f"GT: {GT_FOLDER}")
    print(f"Pred: {PRED_FOLDER}")

    # Lấy danh sách file
    files = [f for f in os.listdir(PRED_FOLDER) if f.endswith('.nii.gz')]
    case_ids = [f.replace('.nii.gz', '') for f in files]

    print(f"Tìm thấy {len(case_ids)} cases. Bắt đầu tính HD95 (Sẽ hơi lâu đấy)...")

    results = []
    # Dùng tqdm để hiện thanh tiến trình
    for case_id in tqdm(case_ids):
        res = process_case(case_id)
        if res:
            results.append(res)

    # Tạo DataFrame và Report
    df = pd.DataFrame(results)

    # Tính trung bình
    summary = {
        "Mean_HD95_WT": df["HD95_WT"].mean(),
        "Mean_HD95_TC": df["HD95_TC"].mean(),
        "Mean_HD95_ET": df["HD95_ET"].mean()
    }

    print("\n" + "="*40)
    print("KẾT QUẢ HD95 (mm) - Thấp hơn là tốt hơn")
    print("="*40)
    print(f"Mean HD95 WT: {summary['Mean_HD95_WT']:.4f}")
    print(f"Mean HD95 TC: {summary['Mean_HD95_TC']:.4f}")
    print(f"Mean HD95 ET: {summary['Mean_HD95_ET']:.4f}")
    print("-" * 40)

    # Lưu file
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Đã lưu kết quả vào: {OUTPUT_CSV}")

Đang đọc dữ liệu từ Drive...
GT: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr
Pred: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres/crossval_results_folds_0_1_2_3_4
Tìm thấy 369 cases. Bắt đầu tính HD95 (Sẽ hơi lâu đấy)...


100%|██████████| 369/369 [1:10:42<00:00, 11.50s/it]


KẾT QUẢ HD95 (mm) - Thấp hơn là tốt hơn
Mean HD95 WT: 5.0697
Mean HD95 TC: 7.2368
Mean HD95 ET: 32.0350
----------------------------------------
Đã lưu kết quả vào: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres/crossval_results_folds_0_1_2_3_4/BraTS_HD95_Metrics.csv


### 3.2 EDL

In [12]:
import numpy as np
import nibabel as nib
import pandas as pd
from medpy.metric.binary import hd95
from tqdm import tqdm

# Đường dẫn folder chứa nhãn gốc (Ground Truth) trên Drive
# Folder này chứa các file: BraTS20_Training_001.nii.gz, ...
GT_FOLDER = "/content/nnUNet_raw/Dataset101_BraTS2020/labelsTr"

# Đường dẫn folder kết quả dự đoán (Prediction) trên Drive
# Đây là folder backup bạn vừa tạo xong
PRED_FOLDER = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer__nnUNetPlans__3d_fullres/crossval_results_folds_0_1_2_3_4"

# Nơi lưu file CSV kết quả (Lưu ngay tại folder dự đoán cho tiện)
OUTPUT_CSV = os.path.join(PRED_FOLDER, "BraTS_HD95_Metrics.csv")

In [13]:
def compute_hd95_safe(pred, gt, spacing):
    """
    Tính HD95 an toàn.
    Nếu 1 trong 2 mask rỗng (không có u), HD95 không xác định (vô cực).
    Trong các cuộc thi BraTS, trường hợp này thường được gán giá trị phạt tối đa (373.13mm).
    """
    # Nếu cả 2 đều rỗng -> Khoảng cách bằng 0 (Đúng tuyệt đối)
    if np.sum(pred) == 0 and np.sum(gt) == 0:
        return 0.0

    # Nếu 1 trong 2 rỗng -> Sai hoàn toàn -> Gán giá trị phạt lớn (Max Distance)
    # Đường chéo hộp sọ người khoảng 373.13 mm
    # Điều này có thể xảy ra với 1 số ca không có ET, mà model lỡ dự đoán có ET. Đối với cuộc thi BRats là rank-> scor
    if np.sum(pred) == 0 or np.sum(gt) == 0:
        return 373.13

    try:
        # voxelspacing: Quan trọng để đổi từ pixel sang mm
        return hd95(pred, gt, voxelspacing=spacing)
    except Exception as e:
        print(f"Lỗi tính HD95: {e}")
        return np.nan

def process_case(case_id):
    try:
        gt_path = os.path.join(GT_FOLDER, f"{case_id}.nii.gz")
        pred_path = os.path.join(PRED_FOLDER, f"{case_id}.nii.gz")

        if not os.path.exists(gt_path):
            print(f"Thiếu GT: {case_id}")
            return None
        if not os.path.exists(pred_path):
            print(f"Thiếu Pred: {case_id}")
            return None

        # Load NIfTI
        gt_nii = nib.load(gt_path)
        pred_nii = nib.load(pred_path)

        # Lấy spacing (kích thước voxel theo mm) từ header
        # Thường là (1.0, 1.0, 1.0) hoặc xấp xỉ
        zoom = gt_nii.header.get_zooms()

        gt_data = gt_nii.get_fdata().astype(np.uint8)
        pred_data = pred_nii.get_fdata().astype(np.uint8)

        # --- Tách vùng (Region Extraction) ---
        # Label 1: NCR/NET, 2: ED, 3: ET

        # WT (Whole Tumor): Label 1 + 2 + 3
        gt_WT = (gt_data > 0)
        pred_WT = (pred_data > 0)

        # TC (Tumor Core): Label 1 + 3
        gt_TC = np.logical_or(gt_data == 1, gt_data == 3)
        pred_TC = np.logical_or(pred_data == 1, pred_data == 3)

        # ET (Enhancing Tumor): Label 3
        gt_ET = (gt_data == 3)
        pred_ET = (pred_data == 3)

        # --- Tính HD95 ---
        h_WT = compute_hd95_safe(pred_WT, gt_WT, zoom)
        h_TC = compute_hd95_safe(pred_TC, gt_TC, zoom)
        h_ET = compute_hd95_safe(pred_ET, gt_ET, zoom)

        return {
            "Case_ID": case_id,
            "HD95_WT": h_WT,
            "HD95_TC": h_TC,
            "HD95_ET": h_ET
        }

    except Exception as e:
        print(f"Error {case_id}: {e}")
        return None

In [14]:
if __name__ == "__main__":
    print(f"Đang đọc dữ liệu từ Drive...")
    print(f"GT: {GT_FOLDER}")
    print(f"Pred: {PRED_FOLDER}")

    # Lấy danh sách file
    files = [f for f in os.listdir(PRED_FOLDER) if f.endswith('.nii.gz')]
    case_ids = [f.replace('.nii.gz', '') for f in files]

    print(f"Tìm thấy {len(case_ids)} cases. Bắt đầu tính HD95 (Sẽ hơi lâu đấy)...")

    results = []
    # Dùng tqdm để hiện thanh tiến trình
    for case_id in tqdm(case_ids):
        res = process_case(case_id)
        if res:
            results.append(res)

    # Tạo DataFrame và Report
    df = pd.DataFrame(results)

    # Tính trung bình
    summary = {
        "Mean_HD95_WT": df["HD95_WT"].mean(),
        "Mean_HD95_TC": df["HD95_TC"].mean(),
        "Mean_HD95_ET": df["HD95_ET"].mean()
    }

    print("\n" + "="*40)
    print("KẾT QUẢ HD95 (mm) - Thấp hơn là tốt hơn")
    print("="*40)
    print(f"Mean HD95 WT: {summary['Mean_HD95_WT']:.4f}")
    print(f"Mean HD95 TC: {summary['Mean_HD95_TC']:.4f}")
    print(f"Mean HD95 ET: {summary['Mean_HD95_ET']:.4f}")
    print("-" * 40)

    # Lưu file
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Đã lưu kết quả vào: {OUTPUT_CSV}")

Đang đọc dữ liệu từ Drive...
GT: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr
Pred: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer__nnUNetPlans__3d_fullres/crossval_results_folds_0_1_2_3_4
Tìm thấy 369 cases. Bắt đầu tính HD95 (Sẽ hơi lâu đấy)...


100%|██████████| 369/369 [1:08:57<00:00, 11.21s/it]


KẾT QUẢ HD95 (mm) - Thấp hơn là tốt hơn
Mean HD95 WT: 5.1461
Mean HD95 TC: 6.2760
Mean HD95 ET: 29.3577
----------------------------------------
Đã lưu kết quả vào: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer__nnUNetPlans__3d_fullres/crossval_results_folds_0_1_2_3_4/BraTS_HD95_Metrics.csv
